# SegResNet Fold -1 Inference on MPS

Runs local inference for the full-training `dice_focal` SegResNet checkpoint on the reserved fold `-1` cases from `data/cv_splits_qc.csv`, saves masks under the model output folder, and reports Dice, surface Hausdorff, Hausdorff, IoU, and volume metrics.

In [14]:
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-cache")
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp/xdg-cache")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

Project root: /Users/ricca/Desktop/Health_Informatics_Internship_2026
Python: /opt/miniconda3/bin/python


In [15]:
import torch

print(f"torch: {torch.__version__}")
print(f"MPS built: {torch.backends.mps.is_built()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

if not torch.backends.mps.is_available():
    raise RuntimeError(
        "MPS is not available in this notebook kernel. Select a Python/conda "
        "environment where torch.backends.mps.is_available() is True."
    )

device = torch.device("mps")
print(f"Using device: {device}")

torch: 2.11.0
MPS built: True
MPS available: True
Using device: mps


In [16]:
import importlib
import scripts.run_local_fold_minus1_inference as local_inference
local_inference = importlib.reload(local_inference)

from scripts.run_local_fold_minus1_inference import (
    DEFAULT_CHECKPOINT,
    DEFAULT_INPUT_DATA,
    DEFAULT_OUTPUT_DIR,
    DEFAULT_SPLIT_CSV,
    filter_readable_cases,
    resolve_project_path,
    select_cases_from_split,
)

checkpoint_path = resolve_project_path(DEFAULT_CHECKPOINT)
output_dir = resolve_project_path(DEFAULT_OUTPUT_DIR)
cases = select_cases_from_split(DEFAULT_INPUT_DATA, DEFAULT_SPLIT_CSV, test_fold=-1)
readable_cases, skipped_cases = filter_readable_cases(cases)

print(f"Checkpoint exists: {checkpoint_path.exists()} | {checkpoint_path}")
print(f"Output directory: {output_dir}")
print(f"Fold -1 cases: {len(cases)}")
print(f"Readable cases: {len(readable_cases)}")
print(f"Skipped unreadable cases: {len(skipped_cases)}")
if skipped_cases:
    for skipped in skipped_cases:
        print(skipped)
print([case.patient_id for case in readable_cases])

Checkpoint exists: True | /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/totalsegmentator_heart_myocardium_visualizations/models/dice_focal/best_metric_model.pth
Output directory: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/totalsegmentator_heart_myocardium_visualizations/models/dice_focal/masks
Fold -1 cases: 28
Readable cases: 28
Skipped unreadable cases: 0
['TAVI_002', 'TAVI_006', 'TAVI_025', 'TAVI_028', 'TAVI_035', 'TAVI_036', 'TAVI_065', 'TAVI_072', 'TAVI_078', 'TAVI_084', 'TAVI_085', 'TAVI_086', 'TAVI_094', 'TAVI_100', 'TAVI_125', 'TAVI_149', 'TAVI_157', 'TAVI_240', 'TAVI_251', 'TAVI_258', 'TAVI_303', 'TAVI_305', 'TAVI_311', 'TAVI_318', 'TAVI_320', 'TAVI_323', 'TAVI_356', 'TAVI_364']


In [17]:
import importlib
import scripts.run_local_fold_minus1_inference as local_inference
local_inference = importlib.reload(local_inference)

local_inference.main([
    "--device", "mps",
    "--skip_existing",
])

Checkpoint: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/totalsegmentator_heart_myocardium_visualizations/models/dice_focal/best_metric_model.pth
Output dir: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/totalsegmentator_heart_myocardium_visualizations/models/dice_focal/masks
Running 28 fold -1 case(s) on device=mps.
Loaded ema_model_state_dict from checkpoint.
Skipping 26 case(s) with existing segmentation_model.nii.gz.


/opt/miniconda3/lib/python3.13/site-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


[TAVI_318] inference
[TAVI_318] Dice=0.8954 IoU=0.8106 surfaceHD=6.08mm HD=6.00mm
[TAVI_320] inference
[TAVI_320] Dice=0.8958 IoU=0.8112 surfaceHD=4.50mm HD=4.47mm
Saved masks and metrics under: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/totalsegmentator_heart_myocardium_visualizations/models/dice_focal/masks


0

In [18]:
import pandas as pd

metrics_path = output_dir / "fold_minus1_metrics.csv"
summary_path = output_dir / "fold_minus1_metrics_summary.csv"

metrics = pd.read_csv(metrics_path)
summary = pd.read_csv(summary_path)

display(summary)
display(metrics.sort_values(["mask_type", "dice"], ascending=[True, False]))

,mask_type,case_count,mean_dice,median_dice,mean_surface_hausdorff_mm,median_surface_hausdorff_mm,mean_surface_hausdorff95_mm,median_surface_hausdorff95_mm,mean_hausdorff_mm,median_hausdorff_mm,...,mean_iou,median_iou,mean_pred_volume_ml,median_pred_volume_ml,mean_ground_truth_volume_ml,median_ground_truth_volume_ml,mean_volume_difference_ml,median_volume_difference_ml,mean_volume_ratio_pred_to_gt,median_volume_ratio_pred_to_gt
0,postprocessed,28,0.882632,0.887463,10.472550,6.240704,3.123136,3.000000,10.445475,6.240704,...,0.790780,0.797693,145.326742,138.375660,143.477403,130.235286,1.849339,2.505696,1.019400,1.015383
1,raw,28,0.872636,0.885020,36.075728,6.894908,8.471562,3.012662,36.050013,6.894908,...,0.776689,0.793756,148.163077,142.183879,143.477403,130.235286,4.685674,2.917408,1.048162,1.016284


,patient_id,fold,mask_type,mask_path,dice,surface_hausdorff_mm,surface_hausdorff95_mm,hausdorff_mm,hausdorff95_mm,iou,pred_volume_ml,ground_truth_volume_ml,volume_difference_ml,volume_ratio_pred_to_gt,pred_voxels,ground_truth_voxels
18,TAVI_084,-1,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.924545,4.080986,2.568786,3.865074,0.513757,0.859678,152.723291,147.489628,5.233664,1.035485,385743,372524
20,TAVI_085,-1,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.921537,3.789062,2.426184,3.750981,0.535854,0.854491,116.811523,110.348291,6.463232,1.058571,271207,256201
10,TAVI_036,-1,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.914010,6.141905,3.000000,6.141905,0.711873,0.841638,121.162582,123.074499,-1.911917,0.984465,398485,404773
34,TAVI_240,-1,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.907810,4.271195,2.971142,4.271195,0.911505,0.831183,161.452255,159.968581,1.483674,1.009275,518197,513435
6,TAVI_028,-1,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.905375,6.000000,3.000000,6.000000,0.912770,0.827109,131.005508,122.723841,8.281667,1.067482,262069,245502
28,TAVI_125,-1,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.903408,4.755804,2.917370,4.755804,0.922553,0.823833,123.226455,122.910801,0.315653,1.002568,386091,385102
16,TAVI_078,-1,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.902141,6.283240,3.000000,6.283240,1.055136,0.821727,133.579042,126.713805,6.865237,1.054179,319956,303512
12,TAVI_065,-1,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.900861,6.000000,3.000000,6.000000,1.160097,0.819607,121.438181,126.723237,-5.285056,0.958295,240622,251094
22,TAVI_086,-1,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.896450,6.198169,3.000000,6.198169,1.042969,0.812333,172.304956,176.969373,-4.664417,0.973643,475200,488064
40,TAVI_303,-1,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.896381,7.266201,3.296424,7.266201,1.366166,0.812219,134.146302,147.292428,-13.146126,0.910748,311454,341976


Main postprocessed masks are saved as:

`notebooks/output/totalsegmentator_heart_myocardium_visualizations/models/dice_focal/masks/<PATIENT_ID>/segmentation_model.nii.gz`

Ground truth copies are saved beside them as `ground_truth_mask.nii.gz` for direct comparison.